# 08. Saving & Loading Models — PyTorch

PyTorch's recommended path is **`state_dict`** (weights only). We also show the full-object pickle and **TorchScript** (`torch.jit`) for deployment.

**Dataset:** `loan_prediction_data.csv` — 614 loan applications, 11 pre-normalised features, binary target `Loan_Status` (1 = approved).

> This is the PyTorch twin of `08_..._keras.ipynb`. Where Keras hides the training loop inside `model.fit`, here we **write the loop explicitly** — that is the main thing to compare.

---

## 1. Setup & imports

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

PyTorch 2.11.0+cpu | device: cpu


## 2. Load the data

In [2]:
import os, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

# Robust load: works whether the notebook runs from its own folder or the parent
CSV = "loan_prediction_data.csv"
if not os.path.exists(CSV):
    CSV = os.path.join("..", "loan_prediction_data.csv")

df = pd.read_csv(CSV)
print("Shape:", df.shape)
df.head()

Shape: (614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,0.0,0.0,0.000000,1.0,0.0,0.070489,0.000000,0.198860,0.74359,1.0,1.0,1.0
1,LP001003,0.0,1.0,0.333333,1.0,0.0,0.054830,0.036192,0.172214,0.74359,1.0,0.0,0.0
2,LP001005,0.0,1.0,0.000000,1.0,1.0,0.035250,0.000000,0.082489,0.74359,1.0,1.0,1.0
3,LP001006,0.0,1.0,0.000000,0.0,0.0,0.030093,0.056592,0.160637,0.74359,1.0,1.0,1.0
4,LP001008,0.0,0.0,0.000000,1.0,0.0,0.072356,0.000000,0.191027,0.74359,1.0,1.0,1.0


## 3. Prepare features & train/test split

Features are already scaled to `[0,1]`; we drop the ID and do a **stratified** split.

In [3]:
# Features are ALREADY normalised to [0,1] in this dataset, so no scaling needed.
# Drop the ID column and separate the target.
X = df.drop(columns=["Loan_ID", "Loan_Status"]).values.astype("float32")
y = df["Loan_Status"].values.astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
N_FEATURES = X_train.shape[1]
print("Train:", X_train.shape, " Test:", X_test.shape, " Features:", N_FEATURES)
print("Train positive rate: %.3f  Test positive rate: %.3f" % (y_train.mean(), y_test.mean()))

Train: (491, 11)  Test: (123, 11)  Features: 11
Train positive rate: 0.686  Test positive rate: 0.691


In [4]:
# Wrap the numpy arrays as float tensors. Target is shaped (N,1) to match the
# sigmoid output and BCELoss.
Xtr = torch.tensor(X_train)
ytr = torch.tensor(y_train).view(-1, 1)
Xte = torch.tensor(X_test)
yte = torch.tensor(y_test).view(-1, 1)
print("Xtr", tuple(Xtr.shape), "| ytr", tuple(ytr.shape))

Xtr (491, 11) | ytr (491, 1)


In [5]:
def plot_history(hist, title=""):
    """Plot train (solid) vs test (dashed) loss and accuracy over epochs."""
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(hist["loss"], label="train")
    ax[0].plot(hist["val_loss"], "--", label="test")
    ax[0].set_title(title + " — Loss"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss")
    ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(hist["accuracy"], label="train")
    ax[1].plot(hist["val_accuracy"], "--", label="test")
    ax[1].set_title(title + " — Accuracy"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy")
    ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 4. Train a model to save

In [6]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(N_FEATURES, 32), nn.ReLU(),
    nn.Linear(32, 1), nn.Sigmoid(),
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3); loss_fn = nn.BCELoss()
loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True)
for _ in range(60):
    model.train()
    for xb, yb in loader:
        opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
model.eval()
with torch.no_grad():
    ref_pred = model(Xte[:5]).ravel().numpy()
print("Reference predictions:", np.round(ref_pred, 5))

Reference predictions: [0.15253 0.83325 0.77717 0.79882 0.83134]


## 5. Option A — `state_dict` (recommended)

Save just the learned tensors. To reload you rebuild the architecture, then `load_state_dict`. This is the portable, version-robust way.

In [7]:
torch.save(model.state_dict(), "loan_model_state.pt")

reloaded = nn.Sequential(
    nn.Linear(N_FEATURES, 32), nn.ReLU(),
    nn.Linear(32, 1), nn.Sigmoid(),
)
reloaded.load_state_dict(torch.load("loan_model_state.pt"))
reloaded.eval()
with torch.no_grad():
    new_pred = reloaded(Xte[:5]).ravel().numpy()
print("Reloaded predictions :", np.round(new_pred, 5))
print("Identical to original?", np.allclose(ref_pred, new_pred))

Reloaded predictions : [0.15253 0.83325 0.77717 0.79882 0.83134]
Identical to original? True


## 6. Option B — full model (pickle)

`torch.save(model)` pickles the whole object — convenient but brittle (it ties the file to your class definitions / file layout). Use for quick experiments only.

In [8]:
torch.save(model, "loan_model_full.pt")
full = torch.load("loan_model_full.pt", weights_only=False)
full.eval()
with torch.no_grad():
    full_pred = full(Xte[:5]).ravel().numpy()
print("Full-object preds    :", np.round(full_pred, 5))
print("Identical to original?", np.allclose(ref_pred, full_pred))

Full-object preds    : [0.15253 0.83325 0.77717 0.79882 0.83134]
Identical to original? True


## 7. Option C — TorchScript (for deployment)

`torch.jit.script` compiles the model to a self-contained artifact that runs without the original Python class — the PyTorch analogue of Keras' SavedModel.

In [9]:
scripted = torch.jit.script(model)
scripted.save("loan_model_scripted.pt")

loaded_ts = torch.jit.load("loan_model_scripted.pt")
loaded_ts.eval()
with torch.no_grad():
    ts_pred = loaded_ts(Xte[:5]).ravel().numpy()
print("TorchScript preds    :", np.round(ts_pred, 5))
print("Identical to original?", np.allclose(ref_pred, ts_pred))

TorchScript preds    : [0.15253 0.83325 0.77717 0.79882 0.83134]
Identical to original? True


## Takeaways
| Method | Saves | Reuse | Notes |
|---|---|---|---|
| `state_dict` | weights only | rebuild arch + `load_state_dict` | **recommended**, portable |
| full pickle | whole object | `torch.load(weights_only=False)` | brittle across refactors |
| TorchScript | compiled graph | `torch.jit.load` | deployment, no Python class needed |

- Always call `model.eval()` before inference so Dropout/BatchNorm behave correctly.
- Keras ↔ PyTorch mapping: `.keras` ≈ full pickle, `.weights.h5` ≈ `state_dict`, SavedModel ≈ TorchScript.
- **Next:** `09_comparison_pytorch`.